Offline teaching language view. Code, existing outputs and configuration retain the accepted reader identity. Fixed filing facts and reference figures are historical teaching inputs; new constructed calculations are not market-strategy reproduction.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). This stage delivers language views; portable dependencies and execution are checked in the later isolated cold start.

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.chdir(ROOT)
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch13_valuation_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Default offline teaching; current empirical French execution SKIPPED (permissions unresolved).")
print("Run directory:", RUN_DIR)


# Chapter 13: valuation for quantitative research

Import the shared `src/quant_course/valuation.py`. Orion, Lyra, valuations, PIT times and market capitalizations are constructed inputs. The Apple section checks SEC filing arithmetic only. No market quotes are downloaded, and no observed valuation or trading return is claimed.

In [ ]:
from datetime import date, datetime, timezone
from pathlib import Path
import sys
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "quant_course" / "valuation.py").is_file():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Cannot locate project root containing src/quant_course/valuation.py")
sys.path.insert(0, str(ROOT / "src"))
from quant_course.valuation import (
    Amount, FinancialVintage, MarketCapSnapshot, constructed_company_cases,
    discounted_cash_flow_value, enterprise_multiple, enterprise_value,
    earnings_yield, equity_from_enterprise_value, fcf_yield, fcff_from_ebit,
    fcfe_from_net_income, gordon_terminal_value, industry_percentile_ranks,
    pit_fcf_yield, price_to_book, price_to_earnings,
    weighted_average_cost_of_capital,
)
print("项目目录：", ROOT)
print("单位规则：金额必须先匹配 currency 与 unit")

## From prices and matching claims to multiples

For two constructed companies, combine price and millions of shares into equity value, then bridge to enterprise value using debt, cash, noncontrolling interest, preferred equity and nonoperating assets. The code checks EV values of 650 and 750, and P/E multiples of 20 and 24.

In [ ]:
cases = constructed_company_cases()
results = {}
for name, case in cases.items():
    cap = case["market_cap"]
    ev = enterprise_value(
        cap, case["debt"], case["cash"], noncontrolling_interest=case["nci"],
        preferred_equity=case["preferred"], nonoperating_assets=case["nonoperating_assets"],
    )
    results[name] = {
        "EV": ev.value,
        "Equity from bridge": equity_from_enterprise_value(
            ev, case["debt"], case["cash"], noncontrolling_interest=case["nci"],
            preferred_equity=case["preferred"], nonoperating_assets=case["nonoperating_assets"],
        ).value,
        "P/E": price_to_earnings(cap, case["net_income"]),
        "P/B": price_to_book(cap, case["book_equity"]),
        "EV/EBITDA": enterprise_multiple(ev, case["ebitda"]),
        "EV/EBIT": enterprise_multiple(ev, case["ebit"]),
        "Earnings yield": earnings_yield(cap, case["net_income"]),
        "FCFE/equity": fcf_yield(case["fcfe"], cap, cash_flow_basis="FCFE", claim_basis="Equity"),
        "FCFF/EV": fcf_yield(case["fcff"], ev, cash_flow_basis="FCFF", claim_basis="EV"),
    }
assert results["Orion"]["EV"] == 650
assert results["Lyra"]["EV"] == 750
assert results["Orion"]["P/E"] == 20
assert results["Lyra"]["P/E"] == 24
results

Orion and Lyra have equal EBITDA but different claims, cash and nonoperating assets. Their EV/EBITDA multiples are about 8.67× and 10×. FCFE/equity and FCFF/EV use matching numerators and denominators.

In [ ]:
display(Image(filename=str(ROOT / "figures" / "a3" / "ch13_ev_bridge.png")))

## From accounting cash flow to DCF

FCFF starts from after-tax operating profit; FCFE starts from common-equity income after financing. All constructed amounts are in USD millions. Check FCFF of 27.5, FCFE of 25, terminal value of 1,030 at year 5 and DCF enterprise value of 860.57. The terminal present value contributes 77.79%.

In [ ]:
fcff = fcff_from_ebit(
    Amount(50, "USD", "USD millions"), 0.25,
    Amount(10, "USD", "USD millions"), Amount(15, "USD", "USD millions"),
    Amount(5, "USD", "USD millions"),
)
fcfe = fcfe_from_net_income(
    Amount(30, "USD", "USD millions"), Amount(10, "USD", "USD millions"),
    Amount(15, "USD", "USD millions"), Amount(5, "USD", "USD millions"),
    Amount(5, "USD", "USD millions"),
)
orion = cases["Orion"]
wacc = weighted_average_cost_of_capital(
    orion["market_cap"], orion["debt"], cost_of_equity=0.10,
    cost_of_debt=0.06, tax_rate=0.25,
)
forecast = [Amount(x, "USD", "USD millions") for x in (40, 45, 50, 55, 60)]
base_wacc, stable_growth = 0.09, 0.03
terminal = gordon_terminal_value(forecast[-1], base_wacc, stable_growth)
dcf_ev = discounted_cash_flow_value(
    forecast, base_wacc, cash_flow_basis="FCFF", discount_rate_basis="WACC",
    terminal_value=terminal,
)
pv_terminal = terminal.value / (1 + base_wacc) ** len(forecast)
assert fcff.value == 27.5 and fcfe.value == 25
assert round(terminal.value, 2) == 1030.00
assert round(dcf_ev.value, 2) == 860.57
assert round(pv_terminal / dcf_ev.value * 100, 2) == 77.79
{
    "constructed FCFF": fcff.value,
    "constructed FCFE": fcfe.value,
    "Orion constructed WACC": wacc,
    "terminal value at year 5": terminal.value,
    "DCF enterprise value": dcf_ev.value,
    "PV terminal share": pv_terminal / dcf_ev.value,
}

In [ ]:
display(Image(filename=str(ROOT / "figures" / "a3" / "ch13_terminal_sensitivity.png")))

Blank cells in the frozen sensitivity plot have growth g≥WACC, outside the convergent growth-perpetuity domain. At equality the denominator is zero; when g>WACC the algebraic quotient has a value, but the underlying infinite discounted cash-flow sum diverges. A high terminal share indicates sensitivity rather than model credibility. Figures are existing teaching assets, not newly generated outputs.

## PIT versions, market times and industry ranks

A reporting-period end is not an availability date. Times, versions and quotes below are constructed. At 2025-03-31 only the FY2024 original that is already public and ingested is eligible; later market values and restatements cannot be backfilled.

In [ ]:
utc = timezone.utc
versions = [
    FinancialVintage(
        issuer_id="SIM-ORION", security_id="SIM-ORION-COMMON", fiscal_period="FY2024",
        period_end=date(2024, 12, 31), available_at=datetime(2025, 2, 15, 8, tzinfo=utc),
        ingested_at=datetime(2025, 2, 16, 10, tzinfo=utc), accession="SIM-V1",
        currency="USD", unit="USD millions", net_income_common=30, common_book_equity=180,
        ebit=50, ebitda=75, free_cash_flow=28, cash_flow_basis="FCFE", industry="Software",
    ),
    FinancialVintage(
        issuer_id="SIM-ORION", security_id="SIM-ORION-COMMON", fiscal_period="FY2024",
        period_end=date(2024, 12, 31), available_at=datetime(2025, 4, 2, 8, tzinfo=utc),
        ingested_at=datetime(2025, 4, 2, 9, tzinfo=utc), accession="SIM-V2",
        currency="USD", unit="USD millions", net_income_common=27, common_book_equity=177,
        ebit=48, ebitda=73, free_cash_flow=25, cash_flow_basis="FCFE", industry="Software",
    ),
]
signal_time = datetime(2025, 3, 31, 20, tzinfo=utc)
caps = [
    MarketCapSnapshot("SIM-ORION-COMMON", signal_time, Amount(600, "USD", "USD millions")),
    MarketCapSnapshot("SIM-ORION-COMMON", datetime(2025, 4, 2, 20, tzinfo=utc),
                      Amount(620, "USD", "USD millions")),
]
value, chosen_version, chosen_cap = pit_fcf_yield(
    versions, caps, security_id="SIM-ORION-COMMON", fiscal_period="FY2024", signal_time=signal_time,
)
assert chosen_version.accession == "SIM-V1"
assert chosen_cap.market_cap.value == 600
assert value == 28 / 600
print({"signal_time": signal_time.isoformat(), "accession": chosen_version.accession,
       "market_cap_as_of": chosen_cap.as_of.isoformat(), "FCFE yield": value})

In [ ]:
late_signal = datetime(2025, 4, 3, 20, tzinfo=utc)
late_value, late_version, late_cap = pit_fcf_yield(
    versions, caps, security_id="SIM-ORION-COMMON", fiscal_period="FY2024", signal_time=late_signal,
)
assert late_version.accession == "SIM-V2"
assert late_cap.market_cap.value == 620
assert late_value == 25 / 620
values = {"A": 0.05, "B": 0.03, "C": 0.03, "LOSS": None, "BANK": 0.08}
industry = {"A": "Software", "B": "Software", "C": "Software",
            "LOSS": "Software", "BANK": "Banks"}
ranks = industry_percentile_ranks(values, industry)
assert ranks["A"] == 1.0 and ranks["B"] == ranks["C"] == 0.25
assert ranks["LOSS"] is None and ranks["BANK"] == 0.5
ranks

The constructed signal time is 2025-03-31 20:00 UTC. Any trade must follow it, and any holding return must begin after that trade takes effect. No return or factor performance is calculated here. At the later cutoff the selected values become 25/620; same-industry ties and excluded observations are handled separately.

## Apple SEC filing arithmetic

The FY2024 reporting period ends on 2024-09-28. Amounts are in USD millions and share counts in thousands. Net income of 93,736, reported diluted EPS of USD 6.08 and weighted-average diluted shares of 15,408,095 thousand give EPS through explicit unit conversion. No unverified market price is inserted to claim P/E. Historical source: [Apple FY2024 Form 10-K](https://www.sec.gov/Archives/edgar/data/320193/000032019324000123/aapl-20240928.htm). Setup print labels identify the project directory and currency/unit matching rule.

In [ ]:
net_income_millions = 93_736
weighted_average_diluted_shares_thousands = 15_408_095
calculated_diluted_eps = net_income_millions * 1_000 / weighted_average_diluted_shares_thousands
reported_diluted_eps = 6.08
assert round(calculated_diluted_eps, 2) == reported_diluted_eps
print(f"SEC-stated EPS={reported_diluted_eps:.2f}; arithmetic from rounded filing rows={calculated_diluted_eps:.4f}")
print("No market price is loaded, so this is not a P/E calculation.")